# บทที่ 10: การปรับปรุงและจัดรูปข้อมูลเพื่อการวิเคราะห์

หลังจากสำรวจข้อมูลด้วย Exploratory Data Analysis แล้ว เราจะทราบว่าข้อมูลมีปัญหาใดบ้าง เช่น

- ชื่อคอลัมน์อ่านหรือใช้งานยาก
- ชนิดข้อมูลไม่ตรงกับความหมาย
- มีค่าว่าง
- ข้อความมีรูปแบบไม่สม่ำเสมอ
- มีข้อมูลซ้ำ
- ยังไม่มีตัวแปรที่ตอบคำถามเชิงวิเคราะห์โดยตรง

กระบวนการปรับข้อมูลให้อยู่ในรูปแบบที่เหมาะกับการใช้งานเรียกว่า **Data Wrangling**

Data Wrangling ไม่ใช่การแก้ข้อมูลแบบสุ่ม แต่เป็นการปรับข้อมูลตามกฎที่มีเหตุผล ตรวจสอบได้ และสามารถทำซ้ำได้

เป้าหมายของบทนี้คือสร้างข้อมูลที่

- มีโครงสร้างสม่ำเสมอ
- มีชนิดข้อมูลเหมาะสม
- มีการจัดการค่าว่างอย่างมีหลักการ
- ลดข้อมูลซ้ำที่ไม่จำเป็น
- มีข้อความและหมวดหมู่เป็นมาตรฐาน
- มีตัวแปรใหม่สำหรับการวิเคราะห์
- พร้อมส่งต่อไปวิเคราะห์หรือสร้างรายงาน

## ผลการเรียนรู้ที่คาดหวัง

เมื่อเรียนจบบทนี้ ผู้เรียนจะสามารถ

1. ปรับชื่อคอลัมน์ ชนิดข้อมูล และข้อความให้อยู่ในรูปแบบมาตรฐานได้
2. จัดการค่าว่างและข้อมูลซ้ำตามกฎที่กำหนดได้
3. สร้างตัวแปรและกลุ่มข้อมูลใหม่สำหรับการวิเคราะห์ได้
4. สร้าง Data Wrangling Pipeline ที่เรียกใช้ซ้ำและตรวจสอบผลได้

## ลำดับเนื้อหา

บทเรียนนี้ประกอบด้วยหัวข้อต่อไปนี้

1. เตรียมข้อมูลตั้งต้น
2. ตรวจสอบข้อมูลก่อนปรับปรุง
3. สร้างสำเนาข้อมูล
4. เปลี่ยนชื่อคอลัมน์
5. ปรับชนิดข้อมูล
6. ปรับข้อความให้เป็นมาตรฐาน
7. ตรวจสอบและจัดการค่าว่าง
8. ตรวจสอบและจัดการข้อมูลซ้ำ
9. สร้างตัวแปรสำหรับการวิเคราะห์
10. สร้าง Flag และจัดกลุ่มข้อมูล
11. เลือก Dataset สุดท้าย
12. ตรวจสอบและเปรียบเทียบผลลัพธ์
13. Data Wrangling Checklist
14. ข้อควรระวัง
15. แบบฝึกหัดท้ายบท

## 1. เตรียมข้อมูลตั้งต้น

เริ่มจาก import pandas และ `Path`

ในบทนี้จะใช้ข้อมูลจากไฟล์

```python
"moac_opsmoac_fragile_farmer.xlsx"
```

โดยอ่านข้อมูลจากหลาย Worksheet แล้วรวมเป็น DataFrame ชื่อ `farmer_raw_df`

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
excel_file = Path(
    "moac_opsmoac_fragile_farmer.xlsx"
)

excel_file

PosixPath('moac_opsmoac_fragile_farmer.xlsx')

ก่อนอ่านข้อมูล ควรตรวจสอบว่าไฟล์มีอยู่จริง

In [3]:
print(
    "File exists:",
    excel_file.exists(),
)

print(
    "Is a file:",
    excel_file.is_file(),
)

File exists: True
Is a file: True


กำหนด Worksheet และคอลัมน์ที่ต้องการอ่าน

In [4]:
target_sheets = [
    "v_cpd_fragile",
    "v_dld_fragile",
    "v_doae_fragile",
]

farmer_usecols = [
    "department_code",
    "department",
    "pid",
    "province_code",
    "province",
    "amphur",
    "tambon",
    "is_farmer",
    "farmer_type",
    "main_occupation",
    "income_in",
    "income_out",
    "debts_in",
    "debts_out",
    "updated_at",
]

farmer_dtype = {
    "department_code": "string",
    "pid": "string",
    "province_code": "string",
}

ตรวจสอบชื่อ Worksheet ก่อนอ่าน เพื่อป้องกันการระบุชื่อผิดหรือ Workbook มีโครงสร้างเปลี่ยนไป

In [5]:
excel_data = pd.ExcelFile(
    excel_file
)

excel_data.sheet_names

['v_cpd_fragile',
 'v_dld_fragile',
 'v_doae_fragile',
 'v_dof_fragile',
 'v_raot_fragile']

In [7]:
farmer_df_list = []

for sheet_name in target_sheets:
    if (
        sheet_name
        not in excel_data.sheet_names
    ):
        print(
            f"Sheet not found: {sheet_name}"
        )
        continue

    temp_df = pd.read_excel(
        excel_data,
        sheet_name=sheet_name,
        usecols=farmer_usecols,
        dtype=farmer_dtype,
    )

    temp_df["source_sheet"] = (
        sheet_name
    )

    farmer_df_list.append(temp_df)

farmer_raw_df = pd.concat(
    farmer_df_list,
    ignore_index=True,
)

farmer_raw_df.head()

,department_code,department,pid,province_code,province,amphur,tambon,is_farmer,farmer_type,main_occupation,income_in,income_out,debts_in,debts_out,updated_at,source_sheet
0,cpd,กรมส่งเสริมสหกรณ์,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
1,cpd,กรมส่งเสริมสหกรณ์,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
2,cpd,กรมส่งเสริมสหกรณ์,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
3,cpd,กรมส่งเสริมสหกรณ์,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
4,cpd,กรมส่งเสริมสหกรณ์,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile


คำว่า `raw` หมายถึงข้อมูลตั้งต้นที่เพิ่งนำเข้าและยังไม่ได้ผ่านการปรับปรุง

แม้ข้อมูลจะถูกรวมจากหลาย Worksheet แล้ว แต่ยังถือเป็นข้อมูลดิบสำหรับ Workflow นี้

In [8]:
farmer_raw_df.shape

(95, 16)

In [9]:
farmer_raw_df[
    "source_sheet"
].value_counts(
    dropna=False
)

source_sheet
v_doae_fragile    70
v_cpd_fragile     15
v_dld_fragile     10
Name: count, dtype: int64

## 2. ตรวจสอบข้อมูลก่อนปรับปรุง

ก่อนแก้ไขข้อมูล ควรบันทึกสภาพเริ่มต้นไว้ก่อน เพื่อให้ทราบว่า

- มีจำนวนแถวและคอลัมน์เท่าใด
- มีค่าว่างในคอลัมน์ใด
- ชนิดข้อมูลเป็นอย่างไร
- มีแถวซ้ำหรือไม่
- ค่าตัวเลขมีช่วงอย่างไร

ขั้นตอนนี้เป็น Baseline สำหรับเปรียบเทียบหลังทำ Data Wrangling

In [10]:
farmer_raw_df.head()

,department_code,department,pid,province_code,province,amphur,tambon,is_farmer,farmer_type,main_occupation,income_in,income_out,debts_in,debts_out,updated_at,source_sheet
0,cpd,กรมส่งเสริมสหกรณ์,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
1,cpd,กรมส่งเสริมสหกรณ์,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
2,cpd,กรมส่งเสริมสหกรณ์,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
3,cpd,กรมส่งเสริมสหกรณ์,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
4,cpd,กรมส่งเสริมสหกรณ์,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile


In [11]:
farmer_raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   department_code  95 non-null     string 
 1   department       95 non-null     str    
 2   pid              95 non-null     string 
 3   province_code    67 non-null     string 
 4   province         10 non-null     object 
 5   amphur           10 non-null     object 
 6   tambon           10 non-null     object 
 7   is_farmer        95 non-null     int64  
 8   farmer_type      95 non-null     str    
 9   main_occupation  21 non-null     object 
 10  income_in        64 non-null     float64
 11  income_out       64 non-null     float64
 12  debts_in         64 non-null     float64
 13  debts_out        64 non-null     float64
 14  updated_at       80 non-null     float64
 15  source_sheet     95 non-null     str    
dtypes: float64(5), int64(1), object(4), str(3), string(3)
memory usage: 12.0+ K

In [12]:
raw_missing_count = (
    farmer_raw_df
    .isna()
    .sum()
)

raw_missing_count

department_code     0
department          0
pid                 0
province_code      28
province           85
amphur             85
tambon             85
is_farmer           0
farmer_type         0
main_occupation    74
income_in          31
income_out         31
debts_in           31
debts_out          31
updated_at         15
source_sheet        0
dtype: int64

In [13]:
raw_duplicate_count = (
    farmer_raw_df
    .duplicated()
    .sum()
)

raw_duplicate_count

np.int64(0)

In [14]:
farmer_raw_df.describe()

,is_farmer,income_in,income_out,debts_in,debts_out,updated_at
count,95.0,64.000000,64.000000,64.000000,64.000000,8.000000e+01
mean,1.0,58787.500000,29378.125000,19015.625000,20750.000000,2.025064e+07
std,0.0,72234.150785,54336.916238,30511.248089,53591.932584,1.193210e+03
min,1.0,0.000000,0.000000,0.000000,0.000000,2.024021e+07
25%,1.0,0.000000,0.000000,0.000000,0.000000,2.025082e+07
50%,1.0,31200.000000,10000.000000,0.000000,0.000000,2.025082e+07
75%,1.0,92500.000000,30000.000000,22500.000000,12000.000000,2.025082e+07
max,1.0,300000.000000,310800.000000,120000.000000,300000.000000,2.025082e+07


ควรบันทึกผลตรวจสอบก่อนปรับปรุงไว้ เพื่อใช้ยืนยันว่าการเปลี่ยนแปลงที่เกิดขึ้นเป็นไปตามที่ตั้งใจ

## 3. สร้างสำเนาข้อมูลก่อนปรับปรุง

ไม่ควรแก้ไข DataFrame ต้นฉบับโดยตรง

ให้สร้างสำเนาด้วย `.copy()` แล้วปรับข้อมูลในสำเนา

In [15]:
farmer_clean_df = (
    farmer_raw_df.copy()
)

farmer_clean_df.head()

,department_code,department,pid,province_code,province,amphur,tambon,is_farmer,farmer_type,main_occupation,income_in,income_out,debts_in,debts_out,updated_at,source_sheet
0,cpd,กรมส่งเสริมสหกรณ์,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
1,cpd,กรมส่งเสริมสหกรณ์,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
2,cpd,กรมส่งเสริมสหกรณ์,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
3,cpd,กรมส่งเสริมสหกรณ์,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
4,cpd,กรมส่งเสริมสหกรณ์,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile


## 4. สร้างสำเนาข้อมูลก่อน cleaning 
หลักปฏิบัติที่ดีคือไม่ควรแก้ไข DataFrame ต้นฉบับโดยตรง 

เราจะสร้างสำเนาชื่อ `clean_df`

In [9]:
farmer_clean_df = farmer_raw_df.copy() 

farmer_clean_df

,department_code,department,pid,province_code,province,amphur,tambon,is_farmer,farmer_type,main_occupation,income_in,income_out,debts_in,debts_out,updated_at,source_sheet
0,cpd,กรมส่งเสริมสหกรณ์,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
1,cpd,กรมส่งเสริมสหกรณ์,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
2,cpd,กรมส่งเสริมสหกรณ์,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
3,cpd,กรมส่งเสริมสหกรณ์,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
4,cpd,กรมส่งเสริมสหกรณ์,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,doae,กรมส่งเสริมการเกษตร,b437149cc5b6a79f8866914fd84b644196ed99ba4f3d62...,96,NaN,NaN,NaN,1,เกษตรกรด้านพืช,ประกอบการเกษตร,200000.0,0.0,0.0,0.0,20250825.0,v_doae_fragile
91,doae,กรมส่งเสริมการเกษตร,b1ce3ba83cfcfc5c21414fe2c496a37b635f56cb4de98c...,61,NaN,NaN,NaN,1,เกษตรกรด้านพืช,NaN,0.0,0.0,0.0,0.0,20250825.0,v_doae_fragile
92,doae,กรมส่งเสริมการเกษตร,5bdabb8ac705d3ba6fab31c9f7f1a48c37455ac548d72a...,53,NaN,NaN,NaN,1,เกษตรกรด้านพืช,ประกอบการเกษตร,200000.0,0.0,100000.0,0.0,20250825.0,v_doae_fragile
93,doae,กรมส่งเสริมการเกษตร,d066b479f17041cb1297e115b3ed0c8e8b724769855a97...,12,NaN,NaN,NaN,1,เกษตรกรด้านพืช,NaN,60000.0,0.0,0.0,0.0,20250825.0,v_doae_fragile


รูปแบบที่แนะนำคือ

```python
raw_df = ข้อมูลตั้งต้น
clean_df = raw_df.copy()
```

การแยก Raw Data และ Clean Data ช่วยให้

- เปรียบเทียบก่อนและหลังได้
- ตรวจสอบ Logic ย้อนหลังได้
- เริ่ม Workflow ใหม่ได้โดยไม่ต้องอ่านไฟล์ซ้ำ
- ลดความเสี่ยงจากการเขียนทับข้อมูลต้นฉบับ

## 4. เปลี่ยนชื่อคอลัมน์

ชื่อคอลัมน์ที่ดีควร

- สื่อความหมาย
- ใช้รูปแบบสม่ำเสมอ
- ไม่มีช่องว่าง
- ไม่ยาวเกินความจำเป็น
- พิมพ์และอ้างอิงในโค้ดได้สะดวก

ในตัวอย่างนี้จะเปลี่ยนชื่อบางคอลัมน์ให้ชัดเจนขึ้น

In [16]:
rename_columns = {
    "pid": "person_id",
    "amphur": "district",
    "tambon": "subdistrict",
    "income_in": (
        "income_agriculture"
    ),
    "income_out": (
        "income_non_agriculture"
    ),
    "debts_in": (
        "debt_agriculture"
    ),
    "debts_out": (
        "debt_non_agriculture"
    ),
}

In [17]:
farmer_clean_df = (
    farmer_clean_df.rename(
        columns=rename_columns
    )
)

farmer_clean_df.head()

,department_code,department,person_id,province_code,province,district,subdistrict,is_farmer,farmer_type,main_occupation,income_agriculture,income_non_agriculture,debt_agriculture,debt_non_agriculture,updated_at,source_sheet
0,cpd,กรมส่งเสริมสหกรณ์,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
1,cpd,กรมส่งเสริมสหกรณ์,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
2,cpd,กรมส่งเสริมสหกรณ์,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
3,cpd,กรมส่งเสริมสหกรณ์,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
4,cpd,กรมส่งเสริมสหกรณ์,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile


หลังเปลี่ยนชื่อ ควรตรวจสอบว่าคอลัมน์ใหม่มีอยู่ครบและชื่อเดิมถูกเปลี่ยนตามที่ตั้งใจ

In [18]:
farmer_clean_df.columns.tolist()

['department_code',
 'department',
 'person_id',
 'province_code',
 'province',
 'district',
 'subdistrict',
 'is_farmer',
 'farmer_type',
 'main_occupation',
 'income_agriculture',
 'income_non_agriculture',
 'debt_agriculture',
 'debt_non_agriculture',
 'updated_at',
 'source_sheet']

การเปลี่ยนชื่อคอลัมน์ไม่ได้เปลี่ยนค่าภายในข้อมูล แต่ช่วยให้โค้ดอ่านง่ายและลดความสับสนในขั้นตอนถัดไป

## 5. ปรับชนิดข้อมูล

ชนิดข้อมูลควรสอดคล้องกับความหมายของคอลัมน์

ตัวอย่างเช่น

- รหัสควรเป็น String
- วันที่ควรเป็น Datetime
- รายได้และหนี้ควรเป็น Numeric
- Flag ควรเป็น Boolean

In [19]:
farmer_clean_df.dtypes

department_code            string
department                    str
person_id                  string
province_code              string
province                   object
district                   object
subdistrict                object
is_farmer                   int64
farmer_type                   str
main_occupation            object
income_agriculture        float64
income_non_agriculture    float64
debt_agriculture          float64
debt_non_agriculture      float64
updated_at                float64
source_sheet                  str
dtype: object

### แปลงคอลัมน์รหัสเป็น String

คอลัมน์ต่อไปนี้เป็นรหัส ไม่ใช่ตัวเลขสำหรับการคำนวณ

- `person_id`
- `province_code`
- `department_code`

In [20]:
code_columns = [
    "person_id",
    "province_code",
    "department_code",
]

for column in code_columns:
    farmer_clean_df[column] = (
        farmer_clean_df[column]
        .astype("string")
    )

### แปลงคอลัมน์วันที่

ใช้ `pd.to_datetime()` และกำหนด `errors="coerce"`

ค่าที่แปลงไม่ได้จะกลายเป็น `NaT`

In [21]:
farmer_clean_df["updated_at"] = (
    pd.to_datetime(
        farmer_clean_df[
            "updated_at"
        ],
        errors="coerce",
    )
)

In [22]:
print(
    farmer_clean_df[
        "updated_at"
    ].dtype
)

print(
    farmer_clean_df[
        "updated_at"
    ].min()
)

print(
    farmer_clean_df[
        "updated_at"
    ].max()
)

datetime64[ns]
1970-01-01 00:00:00.020240212
1970-01-01 00:00:00.020250825


### แปลงคอลัมน์รายได้และหนี้เป็น Numeric

แม้คอลัมน์จะดูเหมือนตัวเลข แต่ควรตรวจสอบและแปลงอย่างชัดเจนก่อนนำไปคำนวณ

In [23]:
numeric_columns = [
    "income_agriculture",
    "income_non_agriculture",
    "debt_agriculture",
    "debt_non_agriculture",
]

for column in numeric_columns:
    farmer_clean_df[column] = (
        pd.to_numeric(
            farmer_clean_df[column],
            errors="coerce",
        )
    )

In [24]:
farmer_clean_df[
    code_columns
    + ["updated_at"]
    + numeric_columns
].dtypes

person_id                         string
province_code                     string
department_code                   string
updated_at                datetime64[ns]
income_agriculture               float64
income_non_agriculture           float64
debt_agriculture                 float64
debt_non_agriculture             float64
dtype: object

`errors="coerce"` ช่วยให้ Workflow ทำงานต่อได้เมื่อพบค่าที่แปลงไม่ได้ แต่ค่าดังกล่าวจะกลายเป็นค่าว่าง

จึงต้องตรวจสอบค่าว่างอีกครั้งหลังการแปลงชนิดข้อมูล

## 6. ปรับข้อความให้เป็นมาตรฐาน

ข้อมูลข้อความอาจมีปัญหา เช่น

- ช่องว่างด้านหน้า
- ช่องว่างด้านหลัง
- รูปแบบตัวพิมพ์ไม่สม่ำเสมอ
- ค่าเดียวกันถูกนับแยกเพราะรูปแบบต่างกัน

ขั้นพื้นฐานคือแปลงเป็น String และใช้ `.str.strip()`

In [25]:
text_columns = [
    "department",
    "province",
    "district",
    "subdistrict",
    "farmer_type",
    "main_occupation",
    "source_sheet",
]

In [26]:
for column in text_columns:
    farmer_clean_df[column] = (
        farmer_clean_df[column]
        .astype("string")
        .str.strip()
    )

ควรตรวจสอบค่าหมวดหมู่หลังปรับข้อความ เพื่อดูว่ายังมีค่าที่สะกดหรือใช้รูปแบบต่างกันหรือไม่

In [27]:
farmer_clean_df[
    "farmer_type"
].value_counts(
    dropna=False
)

farmer_type
เกษตรกรด้านพืช                               70
เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร    15
เกษตรกรผู้เลี้ยงสัตว์                        10
Name: count, dtype: Int64

In [28]:
farmer_clean_df[
    "main_occupation"
].value_counts(
    dropna=False
)

main_occupation
<NA>              74
ประกอบการเกษตร    20
รับจ้างทั่วไป      1
Name: count, dtype: Int64

`.str.strip()` แก้เฉพาะช่องว่างด้านหน้าและด้านหลัง

หากมีปัญหาอื่น เช่น

- การสะกดหลายรูปแบบ
- คำย่อ
- ตัวพิมพ์เล็กและใหญ่
- ชื่อเก่าและชื่อใหม่

อาจต้องสร้าง Mapping Rule เพิ่มเติม โดยควรยืนยันกับเจ้าของข้อมูลก่อน

## 7. ตรวจสอบและจัดการค่าว่าง

ก่อนเติมหรือลบค่าว่าง ควรตรวจสอบก่อนว่า

- คอลัมน์ใดว่าง
- ว่างกี่รายการ
- ค่าว่างมีความหมายอย่างไร
- การเติมค่าจะเปลี่ยนความหมายของข้อมูลหรือไม่

In [29]:
farmer_clean_df.isna().sum()

department_code            0
department                 0
person_id                  0
province_code             28
province                  85
district                  85
subdistrict               85
is_farmer                  0
farmer_type                0
main_occupation           74
income_agriculture        31
income_non_agriculture    31
debt_agriculture          31
debt_non_agriculture      31
updated_at                15
source_sheet               0
dtype: int64

### สร้าง Flag ก่อนเติมค่าว่าง

หากเติมค่าว่างทันที เราจะสูญเสียข้อมูลว่าค่าเดิมเคยว่างหรือไม่

จึงสามารถสร้าง Flag เพื่อเก็บหลักฐานก่อนเติมค่า

In [30]:
for column in numeric_columns:
    flag_column = (
        f"{column}_was_missing"
    )

    farmer_clean_df[
        flag_column
    ] = (
        farmer_clean_df[column]
        .isna()
    )

In [31]:
farmer_clean_df[
    [
        "income_agriculture",
        "income_agriculture_was_missing",
        "debt_agriculture",
        "debt_agriculture_was_missing",
    ]
].head()

,income_agriculture,income_agriculture_was_missing,debt_agriculture,debt_agriculture_was_missing
0,NaN,True,NaN,True
1,NaN,True,NaN,True
2,NaN,True,NaN,True
3,NaN,True,NaN,True
4,NaN,True,NaN,True


### เติมค่าว่างในข้อมูลเชิงหมวดหมู่

สำหรับการฝึกนี้ สมมติว่าค่าว่างในคอลัมน์หมวดหมู่ให้แทนด้วย `"Unknown"`

In [32]:
category_columns = [
    "farmer_type",
    "main_occupation",
    "province",
    "district",
    "subdistrict",
]

In [33]:
for column in category_columns:
    farmer_clean_df[column] = (
        farmer_clean_df[column]
        .fillna("Unknown")
    )

การเติม `"Unknown"` ช่วยให้

- ไม่ต้องลบแถว
- นับค่าว่างเป็นหมวดหมู่ได้
- แยกจากค่าหมวดหมู่จริงได้

อย่างไรก็ตาม ต้องยืนยันก่อนว่า `"Unknown"` เหมาะกับความหมายของข้อมูล

### เติมค่าว่างในข้อมูลตัวเลข

สำหรับการฝึกนี้ สมมติว่าค่าว่างในรายได้และหนี้หมายถึง `0`

สมมติฐานนี้อาจไม่ถูกต้องกับข้อมูลจริงทุกชุด

In [34]:
for column in numeric_columns:
    farmer_clean_df[column] = (
        farmer_clean_df[column]
        .fillna(0)
    )

In [35]:
farmer_clean_df[
    category_columns
    + numeric_columns
].isna().sum()

farmer_type               0
main_occupation           0
province                  0
district                  0
subdistrict               0
income_agriculture        0
income_non_agriculture    0
debt_agriculture          0
debt_non_agriculture      0
dtype: int64

In [36]:
farmer_clean_df[
    numeric_columns
].describe()

,income_agriculture,income_non_agriculture,debt_agriculture,debt_non_agriculture
count,95.000000,95.000000,95.000000,95.000000
mean,39604.210526,19791.578947,12810.526316,13978.947368
std,65305.735820,46589.213579,26537.927390,44950.795545
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000
75%,56000.000000,20000.000000,11000.000000,0.000000
max,300000.000000,310800.000000,120000.000000,300000.000000


หลังเติมค่า ควรตรวจสอบสถิติอีกครั้ง เพราะการเติม `0` อาจทำให้

- ค่าเฉลี่ยลดลง
- ค่าต่ำสุดเปลี่ยน
- การกระจายเปลี่ยน
- จำนวนกลุ่มที่ไม่มีรายได้หรือหนี้เพิ่มขึ้น

ดังนั้น การเติมค่าควรมีเหตุผลและบันทึกกฎไว้อย่างชัดเจน

## 8. ตรวจสอบและจัดการข้อมูลซ้ำ

ข้อมูลซ้ำมีหลายลักษณะ เช่น

- แถวซ้ำกันทุกคอลัมน์
- รหัสบุคคลซ้ำ
- บุคคลเดียวกันมาจากหลายหน่วยงาน
- ข้อมูลรายการเดิมหลาย Version

ขั้นแรกจะตรวจเฉพาะแถวที่ซ้ำกันทุกคอลัมน์

In [37]:
exact_duplicate_count = (
    farmer_clean_df
    .duplicated()
    .sum()
)

exact_duplicate_count

np.int64(0)

จากนั้นตรวจ `person_id` ซ้ำ

In [39]:
missing_person_id_count = (
    farmer_clean_df[
        "person_id"
    ]
    .isna()
    .sum()
)

missing_person_id_count

np.int64(0)

ตัวอย่างนี้ข้อมูลไม่มีแถวและ `person_id` ที่ซ้ำเลย ถือเป็นข้อมูลที่มีคุณภาพ

ในกรณีที่พบข้อมูลซ้ำไม่ควรลบข้อมูลทันที เพราะต้องตอบคำถามก่อนว่า

- บุคคลหนึ่งคนปรากฏในหลายหน่วยงานได้หรือไม่
- แต่ละแถวเป็นคนละเหตุการณ์หรือไม่
- ควรเก็บรายการล่าสุดหรือไม่
- ต้องรวมหรือเชื่อมข้อมูลอย่างไร
- `person_id` มีค่าว่างหรือรูปแบบผิดหรือไม่

การจัดการข้อมูลซ้ำต้องอาศัยทั้งโค้ดและความเข้าใจบริบท

## 9. สร้างตัวแปรสำหรับการวิเคราะห์

หลังปรับชนิดข้อมูลและจัดการค่าว่างแล้ว สามารถสร้างตัวแปรใหม่จากข้อมูลเดิมได้

ตัวแปรใหม่ควร

- มีนิยามชัดเจน
- คำนวณซ้ำได้
- มีความหมายต่อการวิเคราะห์
- มีชื่อที่สื่อความหมาย

### สร้างรายได้รวม

```python
income_agriculture
+ income_non_agriculture
```

In [40]:
farmer_clean_df["total_income"] = (
    farmer_clean_df[
        "income_agriculture"
    ]
    + farmer_clean_df[
        "income_non_agriculture"
    ]
)

### สร้างหนี้รวม

```python
debt_agriculture
+ debt_non_agriculture
```

In [41]:
farmer_clean_df["total_debt"] = (
    farmer_clean_df[
        "debt_agriculture"
    ]
    + farmer_clean_df[
        "debt_non_agriculture"
    ]
)

### สร้างรายได้สุทธิ

ในบทนี้กำหนดอย่างง่ายว่า

```python
total_income - total_debt
```

ตัวแปรนี้เป็นเพียงตัวอย่างสำหรับฝึกสร้าง Feature และไม่ได้แทนฐานะทางเศรษฐกิจทั้งหมด

In [42]:
farmer_clean_df["net_income"] = (
    farmer_clean_df[
        "total_income"
    ]
    - farmer_clean_df[
        "total_debt"
    ]
)

In [43]:
farmer_clean_df[
    [
        "income_agriculture",
        "income_non_agriculture",
        "total_income",
        "debt_agriculture",
        "debt_non_agriculture",
        "total_debt",
        "net_income",
    ]
].head()

,income_agriculture,income_non_agriculture,total_income,debt_agriculture,debt_non_agriculture,total_debt,net_income
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0


หลังสร้างตัวแปรใหม่ ควรตรวจสอบช่วงค่าและสถิติเบื้องต้น

In [44]:
farmer_clean_df[
    [
        "total_income",
        "total_debt",
        "net_income",
    ]
].describe()

,total_income,total_debt,net_income
count,95.000000,95.000000,95.000000
mean,59395.789474,26789.473684,32606.315789
std,84566.353044,55040.003776,79532.514173
min,0.000000,0.000000,-200000.000000
25%,0.000000,0.000000,0.000000
50%,10000.000000,0.000000,0.000000
75%,100000.000000,27500.000000,50000.000000
max,400000.000000,306000.000000,343200.000000


## 10. สร้าง Flag และจัดกลุ่มข้อมูล

Flag คือคอลัมน์ Boolean ที่ระบุว่าแต่ละแถวเข้าเงื่อนไขหรือไม่

ตัวอย่างนี้สร้าง Flag ว่าหนี้รวมมากกว่ารายได้รวมหรือไม่

In [45]:
farmer_clean_df[
    "debt_more_than_income"
] = (
    farmer_clean_df[
        "total_debt"
    ]
    > farmer_clean_df[
        "total_income"
    ]
)

In [46]:
farmer_clean_df[
    [
        "total_income",
        "total_debt",
        "debt_more_than_income",
    ]
].head()

,total_income,total_debt,debt_more_than_income
0,0.0,0.0,False
1,0.0,0.0,False
2,0.0,0.0,False
3,0.0,0.0,False
4,0.0,0.0,False


In [47]:
farmer_clean_df[
    "debt_more_than_income"
].value_counts(
    dropna=False
)

debt_more_than_income
False    87
True      8
Name: count, dtype: int64

### จัดกลุ่มระดับหนี้

กำหนดเกณฑ์ดังนี้

- `"No debt"` เมื่อหนี้รวมเท่ากับ 0
- `"Low debt"` เมื่อหนี้รวมมากกว่า 0 แต่ไม่เกิน 50,000
- `"Medium debt"` เมื่อหนี้รวมมากกว่า 50,000 แต่ไม่เกิน 200,000
- `"High debt"` เมื่อหนี้รวมมากกว่า 200,000

In [48]:
def classify_debt_level(
    total_debt
):
    if total_debt == 0:
        return "No debt"
    elif total_debt <= 50_000:
        return "Low debt"
    elif total_debt <= 200_000:
        return "Medium debt"
    else:
        return "High debt"

In [49]:
farmer_clean_df["debt_level"] = (
    farmer_clean_df[
        "total_debt"
    ]
    .apply(classify_debt_level)
)

In [50]:
farmer_clean_df[
    [
        "total_debt",
        "debt_level",
    ]
].head()

,total_debt,debt_level
0,0.0,No debt
1,0.0,No debt
2,0.0,No debt
3,0.0,No debt
4,0.0,No debt


In [51]:
farmer_clean_df[
    "debt_level"
].value_counts(
    dropna=False
)

debt_level
No debt        63
Medium debt    18
Low debt       12
High debt       2
Name: count, dtype: int64

การจัดกลุ่มช่วยให้สรุปข้อมูลได้ง่ายกว่าการดูตัวเลขดิบเพียงอย่างเดียว

อย่างไรก็ตาม เกณฑ์ทุกช่วงควรมีที่มาและได้รับการยืนยันจากผู้เชี่ยวชาญหรือเจ้าของข้อมูล

## 11. เลือก Dataset สุดท้าย

หลังปรับข้อมูลแล้ว ควรเลือกเฉพาะคอลัมน์ที่จำเป็นต่อการวิเคราะห์

การเลือกคอลัมน์ช่วยให้

- Dataset กระชับ
- ขอบเขตข้อมูลชัดเจน
- ลดคอลัมน์ที่ไม่เกี่ยวข้อง
- ควบคุม Schema ของผลลัพธ์

In [52]:
analysis_columns = [
    "department_code",
    "department",
    "person_id",
    "province_code",
    "province",
    "district",
    "subdistrict",
    "is_farmer",
    "farmer_type",
    "main_occupation",
    "income_agriculture",
    "income_non_agriculture",
    "debt_agriculture",
    "debt_non_agriculture",
    "total_income",
    "total_debt",
    "net_income",
    "debt_more_than_income",
    "debt_level",
    "updated_at",
    "source_sheet",
]

In [53]:
farmer_analysis_df = (
    farmer_clean_df[
        analysis_columns
    ]
    .copy()
)

farmer_analysis_df.head()

,department_code,department,person_id,province_code,province,district,subdistrict,is_farmer,farmer_type,main_occupation,...,income_non_agriculture,debt_agriculture,debt_non_agriculture,total_income,total_debt,net_income,debt_more_than_income,debt_level,updated_at,source_sheet
0,cpd,กรมส่งเสริมสหกรณ์,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,<NA>,Unknown,Unknown,Unknown,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,Unknown,...,0.0,0.0,0.0,0.0,0.0,0.0,False,No debt,NaT,v_cpd_fragile
1,cpd,กรมส่งเสริมสหกรณ์,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,<NA>,Unknown,Unknown,Unknown,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,Unknown,...,0.0,0.0,0.0,0.0,0.0,0.0,False,No debt,NaT,v_cpd_fragile
2,cpd,กรมส่งเสริมสหกรณ์,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,<NA>,Unknown,Unknown,Unknown,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,Unknown,...,0.0,0.0,0.0,0.0,0.0,0.0,False,No debt,NaT,v_cpd_fragile
3,cpd,กรมส่งเสริมสหกรณ์,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,<NA>,Unknown,Unknown,Unknown,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,Unknown,...,0.0,0.0,0.0,0.0,0.0,0.0,False,No debt,NaT,v_cpd_fragile
4,cpd,กรมส่งเสริมสหกรณ์,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,<NA>,Unknown,Unknown,Unknown,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,Unknown,...,0.0,0.0,0.0,0.0,0.0,0.0,False,No debt,NaT,v_cpd_fragile


การใช้ `.copy()` ช่วยให้ `farmer_analysis_df` เป็น Dataset แยกจาก DataFrame ระหว่างการปรับข้อมูล

## 12. ตรวจสอบและเปรียบเทียบผลลัพธ์

ก่อนส่ง Dataset ไปใช้ต่อ ควรตรวจสอบอีกครั้งว่า

- จำนวนแถวสมเหตุสมผล
- คอลัมน์ครบ
- ชนิดข้อมูลถูกต้อง
- ค่าว่างเหลือเท่าใด
- ไม่มีแถวซ้ำที่ไม่ตั้งใจ
- ตัวแปรใหม่มีช่วงค่าที่สมเหตุสมผล

In [54]:
farmer_analysis_df.shape

(95, 21)

In [55]:
farmer_analysis_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   department_code         95 non-null     string        
 1   department              95 non-null     string        
 2   person_id               95 non-null     string        
 3   province_code           67 non-null     string        
 4   province                95 non-null     string        
 5   district                95 non-null     string        
 6   subdistrict             95 non-null     string        
 7   is_farmer               95 non-null     int64         
 8   farmer_type             95 non-null     string        
 9   main_occupation         95 non-null     string        
 10  income_agriculture      95 non-null     float64       
 11  income_non_agriculture  95 non-null     float64       
 12  debt_agriculture        95 non-null     float64       
 13  deb

In [56]:
farmer_analysis_df.isna().sum()

department_code            0
department                 0
person_id                  0
province_code             28
province                   0
district                   0
subdistrict                0
is_farmer                  0
farmer_type                0
main_occupation            0
income_agriculture         0
income_non_agriculture     0
debt_agriculture           0
debt_non_agriculture       0
total_income               0
total_debt                 0
net_income                 0
debt_more_than_income      0
debt_level                 0
updated_at                15
source_sheet               0
dtype: int64

In [57]:
farmer_analysis_df.duplicated().sum()

np.int64(0)

In [58]:
farmer_analysis_df.describe()

,is_farmer,income_agriculture,income_non_agriculture,debt_agriculture,debt_non_agriculture,total_income,total_debt,net_income,updated_at
count,95.0,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,80
mean,1.0,39604.210526,19791.578947,12810.526316,13978.947368,59395.789474,26789.473684,32606.315789,1970-01-01 00:00:00.020250636
min,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-200000.000000,1970-01-01 00:00:00.020240212
25%,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1970-01-01 00:00:00.020250825
50%,1.0,0.000000,0.000000,0.000000,0.000000,10000.000000,0.000000,0.000000,1970-01-01 00:00:00.020250825
75%,1.0,56000.000000,20000.000000,11000.000000,0.000000,100000.000000,27500.000000,50000.000000,1970-01-01 00:00:00.020250825
max,1.0,300000.000000,310800.000000,120000.000000,300000.000000,400000.000000,306000.000000,343200.000000,1970-01-01 00:00:00.020250825
std,0.0,65305.735820,46589.213579,26537.927390,44950.795545,84566.353044,55040.003776,79532.514173,NaN


### เปรียบเทียบขนาดก่อนและหลัง

In [59]:
print(
    "Raw data shape:",
    farmer_raw_df.shape,
)

print(
    "Analysis data shape:",
    farmer_analysis_df.shape,
)

Raw data shape: (95, 16)
Analysis data shape: (95, 21)


จำนวนคอลัมน์อาจเพิ่มหรือลดได้ เนื่องจากมีการ

- เปลี่ยนชื่อคอลัมน์
- สร้างตัวแปรใหม่
- เลือกเฉพาะคอลัมน์สุดท้าย

จำนวนแถวควรเปลี่ยนเฉพาะจากกฎที่ตั้งใจ เช่น การลบแถวซ้ำทุกคอลัมน์

### สร้างตารางสรุปค่าว่างของ Dataset สุดท้าย

In [60]:
final_missing_summary = pd.DataFrame(
    {
        "column": (
            farmer_analysis_df
            .columns
        ),
        "missing_count": (
            farmer_analysis_df
            .isna()
            .sum()
            .values
        ),
        "missing_percent": (
            farmer_analysis_df
            .isna()
            .mean()
            .mul(100)
            .round(2)
            .values
        ),
    }
)

final_missing_summary = (
    final_missing_summary
    .sort_values(
        "missing_percent",
        ascending=False,
    )
    .reset_index(drop=True)
)

final_missing_summary

,column,missing_count,missing_percent
0,province_code,28,29.47
1,updated_at,15,15.79
2,department_code,0,0.00
3,person_id,0,0.00
4,department,0,0.00
5,province,0,0.00
6,district,0,0.00
7,is_farmer,0,0.00
8,subdistrict,0,0.00
9,main_occupation,0,0.00


### สร้างตารางสรุปก่อนและหลัง

In [61]:
wrangling_summary = pd.DataFrame(
    {
        "metric": [
            "row_count",
            "column_count",
            "missing_value_count",
            "exact_duplicate_count",
        ],
        "raw_data": [
            farmer_raw_df.shape[0],
            farmer_raw_df.shape[1],
            (
                farmer_raw_df
                .isna()
                .sum()
                .sum()
            ),
            (
                farmer_raw_df
                .duplicated()
                .sum()
            ),
        ],
        "analysis_data": [
            farmer_analysis_df.shape[0],
            farmer_analysis_df.shape[1],
            (
                farmer_analysis_df
                .isna()
                .sum()
                .sum()
            ),
            (
                farmer_analysis_df
                .duplicated()
                .sum()
            ),
        ],
    }
)

wrangling_summary

,metric,raw_data,analysis_data
0,row_count,95,95
1,column_count,16,21
2,missing_value_count,496,43
3,exact_duplicate_count,0,0


การลดลงของค่าว่างหรือข้อมูลซ้ำไม่ใช่หลักฐานเพียงอย่างเดียวว่าคุณภาพข้อมูลดีขึ้น

ต้องตรวจสอบด้วยว่า

- ความหมายของข้อมูลยังถูกต้อง
- ไม่ได้ลบข้อมูลสำคัญ
- ไม่ได้แทนค่าว่างด้วยค่าที่ทำให้เข้าใจผิด
- ตัวแปรใหม่ใช้สูตรที่ถูกต้อง
- ผลลัพธ์สามารถตรวจสอบย้อนกลับได้

## 13. ข้อควรระวังในการปรับข้อมูล

การปรับข้อมูลทุกขั้นตอนควรมีเหตุผลรองรับ

ข้อควรระวังสำคัญ ได้แก่

- ไม่ลบข้อมูลเพียงเพราะมีค่าว่าง
- ไม่เติมค่าว่างเป็น `0` หากยังไม่ทราบความหมาย
- ไม่ใช้ `"Unknown"` แทนค่าว่างทุกกรณีโดยอัตโนมัติ
- ไม่ลบข้อมูลซ้ำตามรหัสโดยไม่เข้าใจหน่วยของข้อมูล
- ไม่รวมหมวดหมู่ที่ดูคล้ายกันโดยไม่ตรวจความหมาย
- ไม่สร้างตัวแปรใหม่โดยไม่มีนิยามที่ชัดเจน
- ไม่แก้ไข Raw Data โดยตรง
- ควรเก็บกฎการปรับข้อมูลไว้ในโค้ดหรือ Function
- ควรตรวจสอบผลก่อนและหลังทุกครั้ง

# แบบฝึกหัดท้ายบท
แบบฝึกหัดท้ายบทนี้จำลองสถานการณ์การพัฒนาคุณภาพข้อมูลในงานจริง 

ให้ใช้ DataFrame `farmer_raw_df` ที่เตรียมไว้แล้วเป็นข้อมูลตั้งต้น

## แบบฝึกหัดที่ 1: สร้าง function สำหรับปรับชื่อ column และ dtype
ทีมมี `farmer_raw_df` ที่เตรียมมาจากไฟล์ Excel แล้ว 

ขั้นตอนแรกของการพัฒนาคุณภาพข้อมูลคือทำให้ชื่อ column และ dtype 

อยู่ในรูปแบบมาตรฐานเดียวกันทุกครั้ง

### ข้อกำหนด
ให้สร้าง function ชื่อ `standardize_farmer_columns()` โดยมี parameter ดังนี้ 

```python 
def standardize_farmer_columns(df): 
    ... 
```

function นี้ต้องทำงานดังนี้ 
1. สร้างสำเนา DataFrame ด้วย `.copy()` 
2. เปลี่ยนชื่อ column ดังนี้ 
- `pid` เป็น `person_id` 
- `amphur` เป็น `district` 
- `tambon` เป็น `subdistrict` 
- `income_in` เป็น `income_agriculture` 
- `income_out` เป็น `income_non_agriculture` 
- `debts_in` เป็น `debt_agriculture` 
- `debts_out` เป็น `debt_non_agriculture` 
3. แปลง column รหัสต่อไปนี้เป็น string 
- `person_id` 
- `province_code` 
- `department_code` 
4. แปลง `updated_at` เป็น datetime โดยใช้ `pd.to_datetime(errors="coerce")` 
5. return DataFrame ที่ปรับแล้ว หลังจากสร้าง function แล้ว ให้เรียกใช้กับ `farmer_raw_df`

```python 
farmer_step1_df = standardize_farmer_columns(farmer_raw_df) 
```

In [46]:
# เขียนคำตอบของคุณใน Cell นี้

> ### เฉลยแบบฝึกหัดที่ 1
>
> Function ควรสร้างสำเนาก่อนปรับข้อมูล เพื่อไม่ให้เปลี่ยน `farmer_raw_df`
>
> ใช้ `.rename()` ปรับชื่อคอลัมน์ ใช้ `.astype("string")` กับคอลัมน์รหัส และใช้ `pd.to_datetime()` กับวันที่

In [62]:
def standardize_farmer_columns(df):
    clean_df = df.copy()

    rename_columns = {
        "pid": "person_id",
        "amphur": "district",
        "tambon": "subdistrict",
        "income_in": (
            "income_agriculture"
        ),
        "income_out": (
            "income_non_agriculture"
        ),
        "debts_in": (
            "debt_agriculture"
        ),
        "debts_out": (
            "debt_non_agriculture"
        ),
    }

    clean_df = clean_df.rename(
        columns=rename_columns
    )

    code_columns = [
        "person_id",
        "province_code",
        "department_code",
    ]

    for column in code_columns:
        clean_df[column] = (
            clean_df[column]
            .astype("string")
        )

    clean_df["updated_at"] = (
        pd.to_datetime(
            clean_df["updated_at"],
            errors="coerce",
        )
    )

    return clean_df

In [63]:
farmer_step1_df = (
    standardize_farmer_columns(
        farmer_raw_df
    )
)

farmer_step1_df.head()

,department_code,department,person_id,province_code,province,district,subdistrict,is_farmer,farmer_type,main_occupation,income_agriculture,income_non_agriculture,debt_agriculture,debt_non_agriculture,updated_at,source_sheet
0,cpd,กรมส่งเสริมสหกรณ์,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaT,v_cpd_fragile
1,cpd,กรมส่งเสริมสหกรณ์,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaT,v_cpd_fragile
2,cpd,กรมส่งเสริมสหกรณ์,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaT,v_cpd_fragile
3,cpd,กรมส่งเสริมสหกรณ์,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaT,v_cpd_fragile
4,cpd,กรมส่งเสริมสหกรณ์,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaT,v_cpd_fragile


In [64]:
farmer_step1_df[
    [
        "person_id",
        "province_code",
        "department_code",
        "updated_at",
    ]
].dtypes

person_id                  string
province_code              string
department_code            string
updated_at         datetime64[ns]
dtype: object

## แบบฝึกหัดที่ 2: สร้าง function สำหรับจัดการค่าว่างและข้อความ
หลังจากปรับชื่อ column แล้ว ทีมต้องการจัดการค่าว่างและข้อความให้สม่ำเสมอ 

ขั้นตอนนี้ต้องทำซ้ำได้ เพราะเมื่อมีข้อมูลรอบใหม่เข้ามา จะต้องใช้ logic เดิมอีกครั้ง

### ข้อกำหนด
ให้สร้าง function ชื่อ `clean_farmer_missing_and_text()` โดยมี parameter ดังนี้ 

```python 
def clean_farmer_missing_and_text(df): 
    ... 
```

function นี้ต้องทำงานดังนี้ 
1. สร้างสำเนา DataFrame ด้วย `.copy()` 
2. เติมค่าว่างใน column เชิงหมวดหมู่ด้วย `"Unknown"` 
3. เติมค่าว่างใน column ตัวเลขรายได้และหนี้สินด้วย `0` 
4. ใช้ loop และ `.str.strip()` เพื่อตัดช่องว่างหน้าและหลังใน column ข้อความ 
5. return DataFrame ที่ปรับแล้ว 

#### column เชิงหมวดหมู่ที่แนะนำ 

```python 
[ 
    "farmer_type", 
    "main_occupation", 
    "province", 
    "district", 
    "subdistrict" 
] 
```

#### column ตัวเลขที่แนะนำ 

```python 
[ 
    "income_agriculture", 
    "income_non_agriculture", 
    "debt_agriculture", 
    "debt_non_agriculture" 
] 
``` 

#### column ข้อความที่แนะนำ 

```python 
[ 
    "department", 
    "province", 
    "district", 
    "subdistrict", 
    "farmer_type", 
    "main_occupation", 
    "source_sheet" 
] 
``` 

หลังจากสร้าง function แล้ว ให้เรียกใช้กับ `farmer_step1_df` 

```python 
farmer_step2_df = clean_farmer_missing_and_text(farmer_step1_df) 
```

In [47]:
# เขียนคำตอบของคุณใน Cell นี้

> ### เฉลยแบบฝึกหัดที่ 2
>
> ควรแปลงคอลัมน์ตัวเลขด้วย `pd.to_numeric(errors="coerce")` ก่อนเติม `0` เพื่อให้แน่ใจว่าสามารถนำไปคำนวณได้
>
> สำหรับคอลัมน์ข้อความ ให้เติม `"Unknown"` ก่อน แล้วจึงใช้ `.astype("string").str.strip()`

In [65]:
def clean_farmer_missing_and_text(
    df
):
    clean_df = df.copy()

    category_columns = [
        "farmer_type",
        "main_occupation",
        "province",
        "district",
        "subdistrict",
    ]

    numeric_columns = [
        "income_agriculture",
        "income_non_agriculture",
        "debt_agriculture",
        "debt_non_agriculture",
    ]

    text_columns = [
        "department",
        "province",
        "district",
        "subdistrict",
        "farmer_type",
        "main_occupation",
        "source_sheet",
    ]

    for column in numeric_columns:
        clean_df[column] = (
            pd.to_numeric(
                clean_df[column],
                errors="coerce",
            )
        )

    for column in category_columns:
        clean_df[column] = (
            clean_df[column]
            .fillna("Unknown")
        )

    for column in numeric_columns:
        clean_df[column] = (
            clean_df[column]
            .fillna(0)
        )

    for column in text_columns:
        clean_df[column] = (
            clean_df[column]
            .astype("string")
            .str.strip()
        )

    return clean_df

In [66]:
farmer_step2_df = (
    clean_farmer_missing_and_text(
        farmer_step1_df
    )
)

farmer_step2_df.head()

,department_code,department,person_id,province_code,province,district,subdistrict,is_farmer,farmer_type,main_occupation,income_agriculture,income_non_agriculture,debt_agriculture,debt_non_agriculture,updated_at,source_sheet
0,cpd,กรมส่งเสริมสหกรณ์,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,<NA>,Unknown,Unknown,Unknown,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,Unknown,0.0,0.0,0.0,0.0,NaT,v_cpd_fragile
1,cpd,กรมส่งเสริมสหกรณ์,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,<NA>,Unknown,Unknown,Unknown,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,Unknown,0.0,0.0,0.0,0.0,NaT,v_cpd_fragile
2,cpd,กรมส่งเสริมสหกรณ์,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,<NA>,Unknown,Unknown,Unknown,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,Unknown,0.0,0.0,0.0,0.0,NaT,v_cpd_fragile
3,cpd,กรมส่งเสริมสหกรณ์,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,<NA>,Unknown,Unknown,Unknown,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,Unknown,0.0,0.0,0.0,0.0,NaT,v_cpd_fragile
4,cpd,กรมส่งเสริมสหกรณ์,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,<NA>,Unknown,Unknown,Unknown,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,Unknown,0.0,0.0,0.0,0.0,NaT,v_cpd_fragile


In [67]:
farmer_step2_df[
    [
        "farmer_type",
        "main_occupation",
        "province",
        "district",
        "subdistrict",
        "income_agriculture",
        "income_non_agriculture",
        "debt_agriculture",
        "debt_non_agriculture",
    ]
].isna().sum()

farmer_type               0
main_occupation           0
province                  0
district                  0
subdistrict               0
income_agriculture        0
income_non_agriculture    0
debt_agriculture          0
debt_non_agriculture      0
dtype: int64

## แบบฝึกหัดที่ 3: สร้าง function สำหรับจัดการข้อมูลซ้ำ 

ทีมต้องการลบ row ที่ซ้ำกันทุก column 

แต่ยังไม่ต้องลบข้อมูลซ้ำตาม `person_id` 

เพราะคนหนึ่งคนอาจปรากฏได้ในหลาย sheet หรือหลายหน่วยงาน 

### ข้อกำหนด 
ให้สร้าง function ชื่อ `remove_exact_duplicates()` 

โดยมี parameter ดังนี้ 

```python 
def remove_exact_duplicates(df): 
    ... 
```

function นี้ต้องทำงานดังนี้ 
1. สร้างสำเนา DataFrame ด้วย `.copy()` 
2. นับจำนวน row ที่ซ้ำกันทุก column ก่อนลบ 
3. ลบ row ที่ซ้ำกันทุก column ด้วย `drop_duplicates()` 
4. พิมพ์จำนวน row ที่ถูกลบ 
5. return DataFrame หลังลบข้อมูลซ้ำ ฃ

หลังจากสร้าง function แล้ว ให้เรียกใช้กับ `farmer_step2_df`

```python 
farmer_step3_df = remove_exact_duplicates(farmer_step2_df) 
```

In [48]:
# เขียนคำตอบของคุณใน Cell นี้

> ### เฉลยแบบฝึกหัดที่ 3
>
> ใช้ `.duplicated().sum()` นับจำนวนแถวซ้ำก่อนลบ
>
> ใช้ `.drop_duplicates()` ลบเฉพาะแถวที่ซ้ำกันทุกคอลัมน์ และใช้ `.reset_index(drop=True)` สร้าง Index ใหม่

In [68]:
def remove_exact_duplicates(df):
    clean_df = df.copy()

    duplicate_count = (
        clean_df
        .duplicated()
        .sum()
    )

    clean_df = (
        clean_df
        .drop_duplicates()
        .reset_index(drop=True)
    )

    print(
        "Removed duplicate rows:",
        duplicate_count,
    )

    return clean_df

In [69]:
farmer_step3_df = (
    remove_exact_duplicates(
        farmer_step2_df
    )
)

farmer_step3_df.shape

Removed duplicate rows: 0


(95, 16)

Function ไม่ควรลบข้อมูลเพราะข้อมูลซ้ำอาจเกิดจากบริบททางธุรกิจ ไม่ใช่ข้อมูลซ้ำที่ผิดเสมอไป

## แบบฝึกหัดที่ 4: สร้าง function สำหรับสร้างตัวแปรวิเคราะห์
ทีมต้องการสร้างตัวแปรใหม่จากข้อมูลรายได้และหนี้สิน เพื่อใช้วิเคราะห์ความเปราะบางเบื้องต้น

### ข้อกำหนด
ให้สร้าง function ชื่อ `create_farmer_analysis_features()` 

โดยมี parameter ดังนี้ 

```python 
def create_farmer_analysis_features(df): 
    ... 
```

function นี้ต้องทำงานดังนี้ 
1. สร้างสำเนา DataFrame ด้วย `.copy()` 
2. สร้าง `total_income` 
3. สร้าง `total_debt` 
4. สร้าง `net_income` 
5. สร้าง `debt_more_than_income` 
6. สร้าง `debt_level` จากเงื่อนไข 
- `No debt` ถ้า `total_debt == 0` 
- `Low debt` ถ้า `total_debt <= 50000` 
- `Medium debt` ถ้า `total_debt <= 200000` 
- `High debt` ถ้ามากกว่า 200000 
7. return DataFrame ที่มี column ใหม่แล้ว 

หลังจากสร้าง function แล้ว ให้เรียกใช้กับ `farmer_step3_df`

```python 
farmer_step4_df = create_farmer_analysis_features(farmer_step3_df) 
```

In [49]:
# เขียนคำตอบของคุณใน Cell นี้

> ### เฉลยแบบฝึกหัดที่ 4
>
> คำนวณรายได้และหนี้ด้วยการบวกคอลัมน์โดยตรง
>
> สร้าง Boolean Flag ด้วยการเปรียบเทียบทั้งคอลัมน์ และสร้าง Function ย่อยสำหรับจัดกลุ่ม `debt_level`

In [70]:
def create_farmer_analysis_features(
    df
):
    clean_df = df.copy()

    clean_df["total_income"] = (
        clean_df[
            "income_agriculture"
        ]
        + clean_df[
            "income_non_agriculture"
        ]
    )

    clean_df["total_debt"] = (
        clean_df[
            "debt_agriculture"
        ]
        + clean_df[
            "debt_non_agriculture"
        ]
    )

    clean_df["net_income"] = (
        clean_df["total_income"]
        - clean_df["total_debt"]
    )

    clean_df[
        "debt_more_than_income"
    ] = (
        clean_df["total_debt"]
        > clean_df["total_income"]
    )

    def classify_debt_level(
        total_debt
    ):
        if total_debt == 0:
            return "No debt"
        elif total_debt <= 50_000:
            return "Low debt"
        elif total_debt <= 200_000:
            return "Medium debt"
        else:
            return "High debt"

    clean_df["debt_level"] = (
        clean_df["total_debt"]
        .apply(
            classify_debt_level
        )
    )

    return clean_df

In [71]:
farmer_step4_df = (
    create_farmer_analysis_features(
        farmer_step3_df
    )
)

farmer_step4_df[
    [
        "total_income",
        "total_debt",
        "net_income",
        "debt_more_than_income",
        "debt_level",
    ]
].head()

,total_income,total_debt,net_income,debt_more_than_income,debt_level
0,0.0,0.0,0.0,False,No debt
1,0.0,0.0,0.0,False,No debt
2,0.0,0.0,0.0,False,No debt
3,0.0,0.0,0.0,False,No debt
4,0.0,0.0,0.0,False,No debt


## แบบฝึกหัดที่ 5: สร้าง data quality improvement pipeline
หลังจากสร้าง function ย่อยแล้ว ทีมต้องการ function หลักที่เรียกใช้ทุกขั้นตอน 

เพื่อสร้าง dataset พร้อมวิเคราะห์จาก `farmer_raw_df`

### ข้อกำหนด
ให้สร้าง function ชื่อ `prepare_farmer_analysis_dataset()` โดยมี parameter ดังนี้

```python 
def prepare_farmer_analysis_dataset(raw_df): 
    ... 
```

function นี้ต้องทำงานดังนี้ 
1. เรียกใช้ `standardize_farmer_columns()` 
2. เรียกใช้ `clean_farmer_missing_and_text()` 
3. เรียกใช้ `remove_exact_duplicates()`
4. เรียกใช้ `create_farmer_analysis_features()` 
5. เลือกเฉพาะ column สุดท้ายสำหรับวิเคราะห์ 
6. return DataFrame สุดท้าย

column สุดท้ายสำหรับวิเคราะห์ คือ

```python 
analysis_columns = [ 
    "department_code", 
    "department", 
    "person_id", 
    "province_code", 
    "province", 
    "district", 
    "subdistrict", 
    "is_farmer", 
    "farmer_type", 
    "main_occupation", 
    "income_agriculture", 
    "income_non_agriculture", 
    "debt_agriculture", 
    "debt_non_agriculture", 
    "total_income", 
    "total_debt", 
    "net_income", 
    "debt_more_than_income", 
    "debt_level", 
    "updated_at", 
    "source_sheet" 
] 
```

หลังจากสร้าง function แล้ว ให้เรียกใช้ดังนี้ 

```python 
farmer_analysis_df = prepare_farmer_analysis_dataset(farmer_raw_df) 
```

In [51]:
# เขียนคำตอบของคุณใน Markdown Cell นี้

> ### เฉลยแบบฝึกหัดที่ 5
>
> Function หลักทำหน้าที่ควบคุมลำดับ Workflow โดยเรียก Function ย่อยทีละขั้น
>
> หลังสร้าง Feature แล้ว ให้เลือกเฉพาะคอลัมน์ตาม `analysis_columns` และใช้ `.copy()` ก่อนคืนผลลัพธ์

In [72]:
def prepare_farmer_analysis_dataset(
    raw_df
):
    step1_df = (
        standardize_farmer_columns(
            raw_df
        )
    )

    step2_df = (
        clean_farmer_missing_and_text(
            step1_df
        )
    )

    step3_df = (
        remove_exact_duplicates(
            step2_df
        )
    )

    step4_df = (
        create_farmer_analysis_features(
            step3_df
        )
    )

    analysis_columns = [
        "department_code",
        "department",
        "person_id",
        "province_code",
        "province",
        "district",
        "subdistrict",
        "is_farmer",
        "farmer_type",
        "main_occupation",
        "income_agriculture",
        "income_non_agriculture",
        "debt_agriculture",
        "debt_non_agriculture",
        "total_income",
        "total_debt",
        "net_income",
        "debt_more_than_income",
        "debt_level",
        "updated_at",
        "source_sheet",
    ]

    analysis_df = (
        step4_df[
            analysis_columns
        ]
        .copy()
    )

    return analysis_df

In [73]:
farmer_analysis_df = (
    prepare_farmer_analysis_dataset(
        farmer_raw_df
    )
)

farmer_analysis_df.head()

Removed duplicate rows: 0


,department_code,department,person_id,province_code,province,district,subdistrict,is_farmer,farmer_type,main_occupation,...,income_non_agriculture,debt_agriculture,debt_non_agriculture,total_income,total_debt,net_income,debt_more_than_income,debt_level,updated_at,source_sheet
0,cpd,กรมส่งเสริมสหกรณ์,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,<NA>,Unknown,Unknown,Unknown,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,Unknown,...,0.0,0.0,0.0,0.0,0.0,0.0,False,No debt,NaT,v_cpd_fragile
1,cpd,กรมส่งเสริมสหกรณ์,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,<NA>,Unknown,Unknown,Unknown,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,Unknown,...,0.0,0.0,0.0,0.0,0.0,0.0,False,No debt,NaT,v_cpd_fragile
2,cpd,กรมส่งเสริมสหกรณ์,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,<NA>,Unknown,Unknown,Unknown,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,Unknown,...,0.0,0.0,0.0,0.0,0.0,0.0,False,No debt,NaT,v_cpd_fragile
3,cpd,กรมส่งเสริมสหกรณ์,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,<NA>,Unknown,Unknown,Unknown,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,Unknown,...,0.0,0.0,0.0,0.0,0.0,0.0,False,No debt,NaT,v_cpd_fragile
4,cpd,กรมส่งเสริมสหกรณ์,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,<NA>,Unknown,Unknown,Unknown,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,Unknown,...,0.0,0.0,0.0,0.0,0.0,0.0,False,No debt,NaT,v_cpd_fragile


In [74]:
print(
    farmer_analysis_df.shape
)

print(
    farmer_analysis_df
    .duplicated()
    .sum()
)

farmer_analysis_df.info()

(95, 21)
0
<class 'pandas.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   department_code         95 non-null     string        
 1   department              95 non-null     string        
 2   person_id               95 non-null     string        
 3   province_code           67 non-null     string        
 4   province                95 non-null     string        
 5   district                95 non-null     string        
 6   subdistrict             95 non-null     string        
 7   is_farmer               95 non-null     int64         
 8   farmer_type             95 non-null     string        
 9   main_occupation         95 non-null     string        
 10  income_agriculture      95 non-null     float64       
 11  income_non_agriculture  95 non-null     float64       
 12  debt_agriculture        95 non-null     float64     

In [75]:
farmer_analysis_df.isna().sum()

department_code            0
department                 0
person_id                  0
province_code             28
province                   0
district                   0
subdistrict                0
is_farmer                  0
farmer_type                0
main_occupation            0
income_agriculture         0
income_non_agriculture     0
debt_agriculture           0
debt_non_agriculture       0
total_income               0
total_debt                 0
net_income                 0
debt_more_than_income      0
debt_level                 0
updated_at                15
source_sheet               0
dtype: int64

In [76]:
farmer_analysis_df[
    [
        "total_income",
        "total_debt",
        "net_income",
        "debt_more_than_income",
        "debt_level",
    ]
].head()

,total_income,total_debt,net_income,debt_more_than_income,debt_level
0,0.0,0.0,0.0,False,No debt
1,0.0,0.0,0.0,False,No debt
2,0.0,0.0,0.0,False,No debt
3,0.0,0.0,0.0,False,No debt
4,0.0,0.0,0.0,False,No debt
